# Heart-rate sample EDA: grain, cadence and summary agreement
Seventh reviewed table, for Friday October 23, 2026. We query only `heart_rate_sample` and `heart_rate_values`, using PostgreSQL aggregation and a bounded plot rather than loading millions of raw samples into pandas. No database writes or changes to other notebooks are made.

Questions: What does one sample row represent? Are timestamps repeated or outside their reporting periods? What spacing/gaps are actually observed? Do period-level min/max/means agree with the separate summary table?

Working interpretations: `sample_point_sec` appears to be a Unix-second timestamp; `heart_rate` appears to use beats per minute. Names and plausible conversions are not source definitions. Timestamp convention, nominal cadence, sentinel meanings and summary/resting algorithms remain unverified. No clinical thresholds, diagnosis, causation or predictive validation are used.

## 1. Read-only setup and schema
One candidate row is one recorded heart-rate value at a participant/sample timestamp inside a labelled export/reporting period. `additional_id` may identify an export record and repeat across samples; it is not assumed to identify one sample. Shared metadata identify participant/group/type and calendar/reporting window. Grain is tested below.

Every query runs inside an explicit read-only, repeatable-read PostgreSQL transaction. This keeps table comparisons on one snapshot. Server-side scans/sorts may take time, but only aggregate results and at most 1,500 plot bins reach pandas. `.env` and individual records are never displayed. Run all cells in order; the final cell closes the transaction.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, inspect, text
from dotenv import dotenv_values
from IPython.display import display
PROJECT=Path.cwd()
url=dotenv_values(PROJECT/'.env').get('DATABASE_URL')
assert url, 'Configure DATABASE_URL privately.'
engine=create_engine(url)
del url
assert engine.dialect.name=='postgresql', 'These efficient aggregate queries use PostgreSQL syntax.'
connection=engine.connect().execution_options(isolation_level='REPEATABLE READ')
transaction=connection.begin()
connection.execute(text('SET TRANSACTION READ ONLY'))
connection.execute(text("SET LOCAL statement_timeout = '240s'"))
def query(sql): return pd.read_sql(text(sql),connection)
schema=[]
for table in ['heart_rate_sample','heart_rate_values']:
    for c in inspect(connection).get_columns(table):
        schema.append({'table':table,'column':c['name'],'sql_type':str(c['type'])})
display(pd.DataFrame(schema).query("table == 'heart_rate_sample'"))
key=['sdv_user_id','calendar_date','start_measure_period_sec','end_measure_period_sec']


## 2. Inventory and explicit invalidity candidates
Count every column's nulls, recorded zero/negative heart-rate values and numeric ranges. Negative/nonpositive values are not silently assigned sentinel meanings. Positive-only statistics below are an operational sensitivity subset, not a verified validity range. UTC conversions test timestamp plausibility; a different calendar label can reflect timezone conventions.

Compare sample timestamps with reporting bounds using both endpoint equality and strict outside tests. These are diagnostics: without inclusive/exclusive definitions, a timestamp exactly at the end needs separate interpretation.

In [ ]:
inventory=query("""SELECT COUNT(*) AS sample_rows, COUNT(DISTINCT sdv_user_id) AS participants,
 COUNT(DISTINCT additional_id) AS record_ids, COUNT(DISTINCT calendar_date) AS labelled_dates,
 COUNT(*) FILTER (WHERE heart_rate IS NULL) AS null_hr,
 COUNT(*) FILTER (WHERE heart_rate=0) AS zero_hr,
 COUNT(*) FILTER (WHERE heart_rate<0) AS negative_hr,
 COUNT(*) FILTER (WHERE heart_rate>0) AS positive_hr,
 MIN(heart_rate) AS min_recorded_hr, MAX(heart_rate) AS max_recorded_hr,
 MIN(sample_point_sec) AS minimum_sample_sec, MAX(sample_point_sec) AS maximum_sample_sec,
 COUNT(*) FILTER (WHERE sample_point_sec<start_measure_period_sec) AS before_start,
 COUNT(*) FILTER (WHERE sample_point_sec>end_measure_period_sec) AS after_end,
 COUNT(*) FILTER (WHERE sample_point_sec=start_measure_period_sec) AS exactly_at_start,
 COUNT(*) FILTER (WHERE sample_point_sec=end_measure_period_sec) AS exactly_at_end,
 COUNT(*) FILTER (WHERE (to_timestamp(sample_point_sec) AT TIME ZONE 'UTC')::date=calendar_date) AS UTC_sample_date_matches_label,
 COUNT(*) FILTER (WHERE duration_measure_period_sec<=0) AS nonpositive_period_duration,
 COUNT(*) FILTER (WHERE end_measure_period_sec-start_measure_period_sec<>duration_measure_period_sec) AS duration_mismatch_rows
 FROM heart_rate_sample""")
display(inventory)
columns=[c['column'] for c in schema if c['table']=='heart_rate_sample']
null_sql=', '.join('COUNT(*) FILTER (WHERE "'+c+'" IS NULL) AS "'+c+'"' for c in columns)
display(query('SELECT '+null_sql+' FROM heart_rate_sample').T.rename(columns={0:'null_rows'}))
display(query('SELECT heart_rate, COUNT(*) AS records FROM heart_rate_sample WHERE heart_rate<=0 GROUP BY heart_rate ORDER BY heart_rate'))
timestamp_bounds=pd.to_datetime(inventory[['minimum_sample_sec','maximum_sample_sec']].iloc[0],unit='s',utc=True,errors='coerce')
display(timestamp_bounds.to_frame('UTC_plausibility_conversion'))


## 3. Aggregate by exact participant/date/reporting period
SQL returns one aggregate row per participant/date/start/end key. This is a derived period summary, not proof that raw sample keys are unique. Record IDs and metadata variation are counted before choosing any comparison context. Samples from overlapping exports can weight means more than once; raw-row min/max/means are kept as recorded and duplicate risks are checked separately.

We retain both all-numeric and positive-only means/min/max. Nulls are ignored by SQL numeric aggregates and their counts remain explicit. Positive-only values do not silently replace the original aggregates.

In [ ]:
periods=query("""SELECT sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,
 COUNT(*) AS sample_rows,COUNT(heart_rate) AS numeric_samples,
 COUNT(*) FILTER(WHERE heart_rate>0) AS positive_samples,
 COUNT(DISTINCT sample_point_sec) AS distinct_timestamps,
 COUNT(DISTINCT additional_id) AS record_ids, MIN(additional_id) AS record_id_if_unique,
 COUNT(DISTINCT sdv_group_id) AS group_variants,MIN(sdv_group_id) AS group_if_unique,
 COUNT(DISTINCT duration_measure_period_sec) AS duration_variants, MIN(duration_measure_period_sec) AS duration_if_unique,
 COUNT(DISTINCT sdv_data_type) AS data_type_variants,
 MIN(sample_point_sec) AS first_sample_sec,MAX(sample_point_sec) AS last_sample_sec,
 MIN(heart_rate) AS sample_min_all,MAX(heart_rate) AS sample_max_all,AVG(heart_rate) AS sample_mean_all,
 MIN(heart_rate) FILTER(WHERE heart_rate>0) AS sample_min_positive,
 MAX(heart_rate) FILTER(WHERE heart_rate>0) AS sample_max_positive,
 AVG(heart_rate) FILTER(WHERE heart_rate>0) AS sample_mean_positive
 FROM heart_rate_sample
 GROUP BY sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec""")
periods['calendar_date']=pd.to_datetime(periods.calendar_date,errors='coerce')
display(pd.DataFrame([{'derived_periods':len(periods),'periods_with_multiple_record_ids':int(periods.record_ids.gt(1).sum()),
 'periods_with_multiple_groups':int(periods.group_variants.gt(1).sum()),
 'periods_with_multiple_durations':int(periods.duration_variants.gt(1).sum()),
 'periods_with_multiple_data_types':int(periods.data_type_variants.gt(1).sum()),
 'null_derived_period_key_rows':int(periods[key].isna().any(axis=1).sum()),
 'duplicate_participant_date_periods':int(periods.duplicated(['sdv_user_id','calendar_date'],keep=False).sum())}]))
display(periods[['sample_rows','numeric_samples','positive_samples','distinct_timestamps']].describe().T)
display(query("""SELECT COUNT(*) AS record_id_groups,
 COUNT(*) FILTER(WHERE mapped_periods>1) AS record_ids_mapping_to_multiple_participant_periods
 FROM (SELECT additional_id,COUNT(DISTINCT (sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec)) AS mapped_periods
 FROM heart_rate_sample GROUP BY additional_id) x"""))


## 4. Raw-sample keys, exact duplicates and cross-period collisions
Test participant/period/timestamp and participant/timestamp, rather than treating repeated record IDs as duplicate samples. A timestamp can repeat with equal or conflicting values, within one period or across export periods. Exact duplicates include all 12 raw columns; the query counts complete-row identities explicitly, including metadata.

These scans return aggregate counts only. No deduplication is applied to original data. Conflicting values need source resolution; overlapping periods or duplicate exports can create repeated timestamps without identical full rows.

In [ ]:
duplicate_audit=query("""WITH grouped AS (
 SELECT sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,sample_point_sec,
 COUNT(*) AS n,COUNT(DISTINCT heart_rate) AS hr_values,
 COUNT(DISTINCT (additional_id,sdv_group_id,sdv_data_type,duration_measure_period_sec,week_nr,week_day,heart_rate)) AS full_identities
 FROM heart_rate_sample GROUP BY sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,sample_point_sec)
 SELECT COUNT(*) AS period_timestamp_keys,COUNT(*) FILTER(WHERE n>1) AS duplicate_period_timestamp_keys,
 COALESCE(SUM(n-1),0) AS rows_beyond_unique_period_timestamp,
 COUNT(*) FILTER(WHERE hr_values>1) AS period_timestamp_keys_with_conflicting_hr,
 COALESCE(SUM(n-full_identities),0) AS exact_duplicate_rows_after_first FROM grouped""")
display(duplicate_audit)
global_collision=query("""WITH grouped AS (
 SELECT sdv_user_id,sample_point_sec,COUNT(*) AS n,
 COUNT(DISTINCT (calendar_date,start_measure_period_sec,end_measure_period_sec)) AS periods,
 COUNT(DISTINCT heart_rate) AS hr_values FROM heart_rate_sample GROUP BY sdv_user_id,sample_point_sec)
 SELECT COUNT(*) AS participant_timestamp_keys,COUNT(*) FILTER(WHERE n>1) AS repeated_participant_timestamp_keys,
 COALESCE(SUM(n-1),0) AS rows_beyond_unique_participant_timestamp,
 COUNT(*) FILTER(WHERE periods>1) AS timestamps_shared_across_periods,
 COUNT(*) FILTER(WHERE hr_values>1) AS participant_timestamp_keys_with_conflicting_hr FROM grouped""")
display(global_collision)


## 5. Coverage, overlapping windows and observed cadence
Date coverage uses each participant's first-to-last labelled-date span; it excludes unknown enrolment time. First-to-last sample span is not continuous observation. Reporting windows and sample counts cannot establish full-day wear. We report non-24-hour windows, period overlap and aggregate participant coverage, without IDs.

For spacing, repeated timestamps are first collapsed **only in the diagnostic** so a duplicated export does not create a fake zero-second interval. Within-record gaps use participant + record ID + date + period bounds. A separate participant-wide timestamp sequence classifies successive timestamps as same-record, across-record, or ambiguous membership. Across-record gaps are not mixed with within-record cadence. A single timestamp may belong to multiple records, making membership ambiguous. The mode is the most frequently observed interval, not proof of nominal device cadence. Long gaps are missing observations, not proof of nonwear.

In [ ]:
periods['period_hours']=(periods.end_measure_period_sec-periods.start_measure_period_sec)/3600
ordered=periods.sort_values(key)
prior=ordered.groupby('sdv_user_id').end_measure_period_sec.transform(lambda x:x.cummax().shift())
display(periods.period_hours.value_counts().sort_index().rename_axis('reported_hours').to_frame('periods'))
print('Periods starting before a preceding period ends:',int(ordered.start_measure_period_sec.lt(prior).sum()))
coverage=periods.groupby('sdv_user_id').agg(first=('calendar_date','min'),last=('calendar_date','max'),
 observed_dates=('calendar_date','nunique'),periods=('calendar_date','size'),sample_rows=('sample_rows','sum'),
 first_sample=('first_sample_sec','min'),last_sample=('last_sample_sec','max'))
coverage['span_days']=(coverage['last']-coverage['first']).dt.days+1
coverage['missing_dates_within_span']=coverage.span_days-coverage.observed_dates
coverage['observed_date_percent']=100*coverage.observed_dates/coverage.span_days
coverage['first_to_last_sample_span_hours']=(coverage.last_sample-coverage.first_sample)/3600
display(coverage[['observed_dates','periods','sample_rows','span_days','missing_dates_within_span','observed_date_percent','first_to_last_sample_span_hours']].describe().T)
cadence=query("""WITH record_times AS MATERIALIZED (
 SELECT sdv_user_id,additional_id,calendar_date,start_measure_period_sec,end_measure_period_sec,sample_point_sec
 FROM heart_rate_sample WHERE sample_point_sec IS NOT NULL
 GROUP BY sdv_user_id,additional_id,calendar_date,start_measure_period_sec,end_measure_period_sec,sample_point_sec),
 internal AS (SELECT sample_point_sec-LAG(sample_point_sec) OVER(
 PARTITION BY sdv_user_id,additional_id,calendar_date,start_measure_period_sec,end_measure_period_sec ORDER BY sample_point_sec) AS gap_sec FROM record_times),
 user_times AS (SELECT sdv_user_id,sample_point_sec,COUNT(*) AS memberships,
 MIN(ROW(additional_id,calendar_date,start_measure_period_sec,end_measure_period_sec)::text) AS token
 FROM record_times GROUP BY sdv_user_id,sample_point_sec),
 sequential AS (SELECT *,LAG(sample_point_sec) OVER w AS previous_sec,
 LAG(memberships) OVER w AS previous_memberships,LAG(token) OVER w AS previous_token
 FROM user_times WINDOW w AS(PARTITION BY sdv_user_id ORDER BY sample_point_sec)),
 gaps AS (SELECT 'within_export_record' AS gap_kind,gap_sec FROM internal WHERE gap_sec IS NOT NULL
 UNION ALL SELECT CASE WHEN memberships<>1 OR previous_memberships<>1 THEN 'participant_sequence_ambiguous'
 WHEN token=previous_token THEN 'participant_sequence_same_record' ELSE 'participant_sequence_across_records' END,
 sample_point_sec-previous_sec FROM sequential WHERE previous_sec IS NOT NULL)
 SELECT gap_kind,COUNT(*) AS intervals,MIN(gap_sec) AS min_sec,MAX(gap_sec) AS max_sec,
 AVG(gap_sec) AS mean_sec,MODE() WITHIN GROUP(ORDER BY gap_sec) AS observed_mode_sec,
 PERCENTILE_CONT(0.5) WITHIN GROUP(ORDER BY gap_sec) AS median_sec,
 PERCENTILE_CONT(0.9) WITHIN GROUP(ORDER BY gap_sec) AS p90_sec,
 PERCENTILE_CONT(0.99) WITHIN GROUP(ORDER BY gap_sec) AS p99_sec,
 COUNT(*) FILTER(WHERE gap_sec<=15) AS up_to_15_sec,
 COUNT(*) FILTER(WHERE gap_sec>15 AND gap_sec<=60) AS over_15_to_60_sec,
 COUNT(*) FILTER(WHERE gap_sec>60 AND gap_sec<=300) AS over_60_to_300_sec,
 COUNT(*) FILTER(WHERE gap_sec>300 AND gap_sec<=3600) AS over_300_to_3600_sec,
 COUNT(*) FILTER(WHERE gap_sec>3600) AS over_3600_sec FROM gaps GROUP BY gap_kind""")
display(cadence)
fig,axes=plt.subplots(1,2,figsize=(12,4))
periods.sample_rows.plot.hist(bins=30,ax=axes[0]); axes[0].set(title='Recorded samples per reporting period',xlabel='Sample rows',ylabel='Periods')
bins=['up_to_15_sec','over_15_to_60_sec','over_60_to_300_sec','over_300_to_3600_sec','over_3600_sec']
internal=cadence.loc[cadence.gap_kind.eq('within_export_record'),bins]
if not internal.empty:
    axes[1].bar(['≤15','15–60','60–300','300–3600','>3600'],internal.iloc[0])
    axes[1].set(title='Within-record observed timestamp gaps',xlabel='Seconds (right endpoint included)',ylabel='Intervals'); axes[1].set_yscale('symlog',linthresh=1)
plt.tight_layout(); plt.show()


## 6. Bounded distribution plot
Return a frequency table of recorded heart-rate values, capped at 1,500 distinct bins for plotting; if truncated, the notebook says so. Counts describe recorded samples and therefore weight densely recorded periods more. They are not time-weighted estimates and do not measure population health.

In [ ]:
distribution=query('SELECT heart_rate,COUNT(*) AS samples FROM heart_rate_sample GROUP BY heart_rate ORDER BY heart_rate NULLS FIRST LIMIT 1501')
truncated=len(distribution)>1500
print('Distribution plot truncated to 1,500 bins:',truncated)
plot_data=distribution.head(1500).dropna(subset=['heart_rate'])
fig,ax=plt.subplots(figsize=(9,4)); ax.bar(plot_data.heart_rate,plot_data.samples,width=1)
ax.set(title='Recorded heart-rate sample frequency',xlabel='Recorded value; units unverified',ylabel='Sample rows')
plt.tight_layout(); plt.show()


## 7. Compare sample aggregates with the summary table using period keys
Require unique/non-null derived sample-period and summary keys, then use an outer one-to-one join. Compare record ID/group/duration only where metadata are internally unique. Unmatched keys remain explicit; no daily join or raw-sample/summary join multiplies rows.

Compare all-numeric and positive-only raw-row min/max/means with corresponding summary fields, using complete pairs. Exact mean equality is not expected without knowing rounding, averaging weights, sample filtering or summary algorithms. Absolute mean difference ≤0.5 is a descriptive rounding-scale check, not proof of the algorithm. A plain sample mean is row-weighted, not time-weighted. Resting is not reconstructed because its source definition remains unknown.

In [ ]:
summary=query('SELECT * FROM heart_rate_values')
summary['calendar_date']=pd.to_datetime(summary.calendar_date,errors='coerce')
def valid_key(df,k): return not df[k].isna().any(axis=None) and not df.duplicated(k).any()
display(pd.DataFrame([{'table':name,'rows':len(df),'null_key_rows':int(df[key].isna().any(axis=1).sum()),
 'duplicate_key_rows':int(df.duplicated(key,keep=False).sum()),'valid_unique_key':valid_key(df,key)} for name,df in [('derived_sample_periods',periods),('heart_rate_values',summary)]]))
comparison_rows=[]
if valid_key(periods,key) and valid_key(summary,key):
    j=periods.merge(summary,on=key,how='outer',indicator=True,validate='one_to_one')
    matched=j._merge.eq('both')
    display(j._merge.value_counts().rename_axis('period_join_status').to_frame('periods'))
    context=matched & j.record_ids.eq(1) & j.group_variants.eq(1) & j.duration_variants.eq(1)
    display(pd.DataFrame([{'matched_periods':int(matched.sum()),'unique_context_periods':int(context.sum()),
      'equal_record_ids':int((context & j.record_id_if_unique.eq(j.additional_id)).sum()),
      'equal_groups':int((context & j.group_if_unique.eq(j.sdv_group_id)).sum()),
      'equal_durations':int((context & j.duration_if_unique.eq(j.duration_measure_period_sec)).sum())}]))
    usable=context & j.record_id_if_unique.eq(j.additional_id) & j.group_if_unique.eq(j.sdv_group_id) & j.duration_if_unique.eq(j.duration_measure_period_sec)
    print('Matched periods excluded from comparisons because context is ambiguous/different:',int((matched & ~usable).sum()))
    for subset in ['all','positive']:
        for stat,summary_field in [('min','min_heart_rate_min'),('max','max_heart_rate_min'),('mean','avg_heart_rate_min')]:
            sample_field='sample_'+stat+'_'+subset
            complete=usable & j[sample_field].notna() & j[summary_field].notna()
            delta=j.loc[complete,sample_field]-j.loc[complete,summary_field]
            comparison_rows.append({'sample_subset':subset,'statistic':stat,'complete_pairs':int(complete.sum()),
              'missing_either_among_usable':int((usable & ~complete).sum()),'exactly_equal':int(delta.eq(0).sum()),
              'within_absolute_0_5':int(delta.abs().le(.5).sum()),'within_absolute_1':int(delta.abs().le(1).sum()),
              'median_signed_difference':delta.median(),'median_absolute_difference':delta.abs().median(),
              'maximum_absolute_difference':delta.abs().max()})
    display(pd.DataFrame(comparison_rows))
    d=j.loc[usable & j.sample_mean_positive.notna() & j.avg_heart_rate_min.notna()]
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].scatter(d.avg_heart_rate_min,d.sample_mean_positive,s=15,alpha=.4)
    if len(d):
        lo=min(d.avg_heart_rate_min.min(),d.sample_mean_positive.min()); hi=max(d.avg_heart_rate_min.max(),d.sample_mean_positive.max())
        axes[0].plot([lo,hi],[lo,hi],color='gray',linestyle='--')
    axes[0].set(title='Matched periods: positive-sample mean',xlabel='Recorded summary average',ylabel='Row-weighted sample mean')
    (d.sample_mean_positive-d.avg_heart_rate_min).plot.hist(bins=30,ax=axes[1])
    axes[1].set(title='Sample mean minus summary average',xlabel='Difference in recorded units',ylabel='Complete matched periods')
    plt.tight_layout(); plt.show()
else: print('Comparison skipped: resolve null or duplicated period keys.')


## Verified findings from this execution

- **4,912,004 sample rows, 960 reporting periods/record IDs, eight participants**. All 12 columns are non-null. Heart-rate values are positive, ranging **30–202** in unverified recorded units; no zero/negative sentinel candidates occur. Reporting-period metadata are internally consistent within each derived period.
- Participant/date/period/timestamp keys have **no collisions**, and there are **no exact duplicate full rows**. Each record ID maps to one participant/period, so record ID + sample timestamp also identifies a sample in this snapshot. Participant/timestamp alone is **not unique**: **480 timestamps are shared across periods**, with agreeing heart-rate values and no conflicting recorded values. Do not globally pool overlapping exports without an explicit duplicate policy.
- All samples fall between their recorded period start and end; **810 equal the end** and none equals the start. End-minus-start equals reported duration throughout. **948 periods are exactly 24 hours**, **12** have other durations, and **two overlapping windows** are detected. Endpoint inclusivity and timezone/date conventions remain unverified. About **4.55 million samples** have a UTC date equal to the calendar label; the rest are not automatically errors because the label may follow another timezone.
- Period sample counts range **4–6,000**, with median **5,664**. Participants have **15–250 observed dates** and **61,352–1,387,204 sample rows** each. Observed-date coverage within individual first-to-last labelled-date spans ranges **93.04%–100%**. These are recording/date diagnostics, not wear-time or enrolment coverage.
- **4,911,044 within-export-record timestamp intervals** are observed. Mode, median, 90th and 99th percentiles are **15 seconds**; the maximum gap is **57,975 seconds**. There are **227 within-record gaps above 3,600 seconds**. Thus a common observed interval does not establish continuous sampling.
- The distinct participant-wide sequence contains **950 unambiguous across-record intervals** (126 above one hour, maximum **576,735 seconds**) and **482 intervals with ambiguous record membership** near shared timestamps. These are separate from within-record cadence. Long gaps are not labelled nonwear or inactivity.
- Exact participant/date/start/end outer join to `heart_rate_values`: **960 matches, no sample-only periods, one summary-only period**. The unmatched summary is not recoded as zero. All 960 matched periods have unique and agreeing record ID/group/duration context.
- Sample **minima and maxima exactly match all 960 paired summaries**. All-numeric and positive-only results are identical because every sample is positive. Sample means match exactly in **5/960** periods; **747/960** lie within 0.5 recorded units and **822/960** within one unit. **138** therefore differ by more than one unit. Median absolute difference is **0.320**, maximum **18.458**; median signed sample-minus-summary difference is **−0.127**. No rounding, weighting or filtering algorithm is inferred from this agreement/difference pattern. Resting summaries are not reconstructed.

**Teacher-facing takeaway:** “The sample table contains many observations per export period, with overlapping periods sharing some timestamps. Most observed spacing is 15 seconds, but sizeable gaps occur. Period minima and maxima agree with the summary table, while the plain sample mean sometimes differs substantially, so the summary's averaging rules need confirmation.”

All checks used a single read-only repeatable-read snapshot, and the connection closed successfully. Only aggregate tables and bounded frequency plots were returned to pandas; raw millions of sample rows were not loaded. This is descriptive EDA, not clinical interpretation, causation or model validation.


## 8. Teacher-facing takeaway and limitations
“I first separated sample rows from export records and reporting periods. SQL counted missing/duplicate timestamps and spacing without loading the whole sample stream. I then matched period aggregates to the summary table using validated keys and checked min/max/mean agreement with explicit denominators.”

The observed interval distribution does not establish a nominal sampling schedule or nonwear. Duplicate timestamps, overlapping exports, irregular gaps and unequal coverage can bias raw-row means. Date coverage and first-to-last spans do not establish wear time. Partial periods are not full days. Matched records and arithmetic agreement do not independently validate device accuracy or the summary algorithm. Resting semantics remain unknown; no resting estimate is reconstructed. Sample values and summary fields are interpreted only as recorded, without clinical thresholds or causal claims.

Ask the owner for timestamp/timezone and boundary conventions, nominal/adaptive cadence, export/record ID meaning, sentinel codes, duplicate handling, summary sampling/rounding/weighting rules and resting algorithm. CSV export, RAG and model building remain deferred.

In [ ]:
# Release the read-only snapshot even though it made no database changes.
transaction.rollback()
connection.close()
engine.dispose()
print('Read-only analysis complete; connection closed.')
